# GPU Runtime Diagnostics

This notebook isolates why the final assignment notebooks may train on CPU even when an NVIDIA RTX 3080 is installed.

Run this notebook with the same Jupyter kernel used for the final assignment notebooks. The checks are intentionally diagnostic only: they do not install packages, modify datasets, save model artifacts, or change the assignment notebooks.

## What This Notebook Tests

- Whether the active notebook kernel is the workspace `.venv` Python environment.
- Whether Windows can see the NVIDIA GPU through `nvidia-smi`.
- Whether CUDA-capable Python libraries are installed in the active kernel.
- Whether PyTorch, if installed, can allocate CUDA tensors and train a tiny model on the GPU.
- Why scikit-learn models in the final notebooks still use CPU even when a GPU is present.

In [ ]:
from pathlib import Path
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import warnings

import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    def display(value):
        print(value)
    class Markdown(str):
        pass

pd.set_option("display.max_colwidth", 120)


def find_project_root(start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Final Assignment").exists():
            return candidate
    return current


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "Final Assignment" / "notebooks"
DATA_PATH = PROJECT_ROOT / "Datasets" / "Global Urban Air Quality & Pollution Time-Series" / "global_urban_smog_pm25_hourly.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Dataset exists: {DATA_PATH.exists()}")

## 1. Active Runtime and Environment Variables

If this table shows a different Python executable than the workspace `.venv`, the notebook kernel is not using the same environment that the assignment notebooks expect.

In [ ]:
def command_result(args, timeout=20):
    try:
        completed = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
        return {
            "ok": completed.returncode == 0,
            "returncode": completed.returncode,
            "stdout": completed.stdout.strip(),
            "stderr": completed.stderr.strip(),
        }
    except FileNotFoundError as exc:
        return {"ok": False, "returncode": None, "stdout": "", "stderr": str(exc)}
    except subprocess.TimeoutExpired as exc:
        return {"ok": False, "returncode": None, "stdout": exc.stdout or "", "stderr": f"Timed out after {timeout}s"}


workspace_python = PROJECT_ROOT / ".venv" / "Scripts" / "python.exe"
active_runtime = pd.DataFrame([
    {"check": "sys.executable", "value": sys.executable},
    {"check": "expected workspace python", "value": str(workspace_python)},
    {"check": "uses workspace .venv", "value": Path(sys.executable).resolve() == workspace_python.resolve()},
    {"check": "python version", "value": sys.version.replace("\n", " ")},
    {"check": "platform", "value": platform.platform()},
    {"check": "current working directory", "value": os.getcwd()},
    {"check": "VIRTUAL_ENV", "value": os.environ.get("VIRTUAL_ENV", "<not set>")},
    {"check": "CUDA_VISIBLE_DEVICES", "value": os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>")},
    {"check": "AML_BALANCED_BACKUP", "value": os.environ.get("AML_BALANCED_BACKUP", "<not set>")},
    {"check": "AML_FAST_SMOKE", "value": os.environ.get("AML_FAST_SMOKE", "<not set>")},
])
display(active_runtime)

## 2. NVIDIA Driver Visibility

`nvidia-smi` confirms whether the operating system and NVIDIA driver can see the GPU. This does not prove that Python training code can use CUDA; it only proves the driver layer is visible.

In [ ]:
nvidia_smi = command_result(["nvidia-smi"], timeout=20)
print(nvidia_smi["stdout"] or nvidia_smi["stderr"])

query = command_result([
    "nvidia-smi",
    "--query-gpu=name,driver_version,memory.total,memory.used,utilization.gpu",
    "--format=csv,noheader",
], timeout=20)

if query["ok"]:
    gpu_rows = []
    for line in query["stdout"].splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) == 5:
            gpu_rows.append({
                "name": parts[0],
                "driver_version": parts[1],
                "memory_total": parts[2],
                "memory_used": parts[3],
                "gpu_utilization": parts[4],
            })
    display(pd.DataFrame(gpu_rows))
else:
    display(pd.DataFrame([{"status": "nvidia-smi failed", "detail": query["stderr"]}]))

nvcc_path = shutil.which("nvcc")
if nvcc_path:
    nvcc = command_result([nvcc_path, "--version"], timeout=20)
    print("\nnvcc path:", nvcc_path)
    print(nvcc["stdout"] or nvcc["stderr"])
else:
    print("\nnvcc was not found on PATH. This is normal for PyTorch wheels because they bundle CUDA runtime libraries.")

## 3. CUDA-Capable Python Package Availability

A visible GPU is not enough. The active Python environment needs a CUDA-enabled library. The final notebooks currently use scikit-learn for most models and PyTorch only for the optional neural-network section.

In [ ]:
package_purposes = {
    "sklearn": "Current CPU-based baseline models in the final notebooks",
    "torch": "PyTorch neural-network section; can use CUDA only with a CUDA-enabled PyTorch build",
    "tensorflow": "Alternative deep-learning backend; not used by the current final notebooks",
    "xgboost": "Can use GPU with the right package/settings, but not currently installed/used",
    "lightgbm": "Can use GPU with the right build/settings, but not currently installed/used",
    "cupy": "CUDA NumPy-like arrays; useful for diagnostics but not current notebook training",
    "cuml": "RAPIDS GPU ML; generally not the easiest Windows-native route",
    "numba": "Can compile CUDA kernels if installed/configured; not current notebook training",
}

package_rows = []
for module_name, purpose in package_purposes.items():
    installed = importlib.util.find_spec(module_name) is not None
    version = None
    if installed:
        try:
            package_name = "scikit-learn" if module_name == "sklearn" else module_name
            version = importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            version = "installed; package metadata not found"
    package_rows.append({
        "module": module_name,
        "installed_in_active_kernel": installed,
        "version": version or "<missing>",
        "purpose": purpose,
    })

display(pd.DataFrame(package_rows))

## 4. Static Scan of the Final Assignment Notebooks

This check explains what the current notebooks are written to use. `n_jobs=-1` means scikit-learn can use CPU cores in parallel; it does not mean GPU acceleration.

In [ ]:
scan_patterns = {
    "imports_torch": "import torch",
    "checks_cuda": "torch.cuda.is_available",
    "configures_cuda_device": 'torch.device("cuda',
    "gpu_model_enabled": "RUN_GPU_NEURAL_MODEL = True",
    "moves_model_or_data_to_device": ".to(device)",
    "trains_torch_model": "optimizer.step()",
    "balanced_backup_flag": "RUN_BALANCED_BACKUP",
    "uses_sklearn": "from sklearn",
    "uses_cpu_parallel_jobs": "n_jobs=-1",
    "mentions_xgboost": "xgboost",
    "mentions_lightgbm": "lightgbm",
}

scan_rows = []
for notebook_path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):
    if notebook_path.name == "00_gpu_runtime_diagnostics.ipynb":
        continue
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    text = "\n".join(
        "".join(cell.get("source", []))
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    )
    row = {"notebook": notebook_path.name}
    for column, pattern in scan_patterns.items():
        row[column] = pattern in text
    scan_rows.append(row)

scan_df = pd.DataFrame(scan_rows)
display(scan_df)

if len(scan_df):
    display(Markdown(
        "**Interpretation:** scikit-learn estimators in these notebooks run on CPU. "
        "A real PyTorch GPU path must enable the model, configure CUDA, move tensors/model to the device, "
        "and execute optimizer steps. A CUDA availability check by itself does not train on the GPU."
    ))

## 5. Root-Cause Decision Table

Read this table after running the first four sections. It converts the environment evidence into likely causes.

In [ ]:
def installed(module_name):
    return importlib.util.find_spec(module_name) is not None

root_cause_rows = []

def add_cause(check, evidence, impact, action):
    root_cause_rows.append({
        "check": check,
        "evidence": evidence,
        "impact": impact,
        "recommended_action": action,
    })

add_cause(
    "GPU visible to Windows",
    "yes" if nvidia_smi["ok"] else f"no: {nvidia_smi['stderr']}",
    "If yes, the driver is probably not the main blocker.",
    "Keep driver installed; continue checking Python CUDA libraries.",
)

add_cause(
    "Active kernel uses workspace .venv",
    str(Path(sys.executable).resolve() == workspace_python.resolve()),
    "If false, installs in `.venv` will not affect this notebook run.",
    "Select the `.venv` kernel in VS Code/Jupyter before rerunning diagnostics.",
)

add_cause(
    "PyTorch installed",
    str(installed("torch")),
    "If false, the final notebooks cannot run their PyTorch GPU neural-network sections.",
    "Install a CUDA-enabled PyTorch wheel in `.venv`, then rerun this notebook.",
)

add_cause(
    "scikit-learn GPU support",
    "scikit-learn installed; standard estimators are CPU-bound",
    "RandomForest, LogisticRegression, SVM, Ridge, HistGradientBoosting, KMeans, and permutation importance use CPU here.",
    "Expect CPU usage for scikit-learn sections; use PyTorch/XGBoost/LightGBM GPU paths only where explicitly implemented.",
)

backup_value = os.environ.get("AML_BALANCED_BACKUP", "<not set>")
add_cause(
    "Backup mode forcing CPU",
    f"AML_BALANCED_BACKUP={backup_value}",
    "The final notebooks set `DEVICE='cpu'` when balanced backup mode is enabled.",
    "Unset `AML_BALANCED_BACKUP` or set it to `0` for GPU-capable PyTorch sections.",
)

cuda_visible = os.environ.get("CUDA_VISIBLE_DEVICES", "<not set>")
add_cause(
    "CUDA devices hidden",
    f"CUDA_VISIBLE_DEVICES={cuda_visible}",
    "An empty or invalid value can hide the GPU from CUDA libraries.",
    "Unset `CUDA_VISIBLE_DEVICES` unless you deliberately need to mask GPUs.",
)

display(pd.DataFrame(root_cause_rows))

## 6. PyTorch CUDA Probe

This section is skipped cleanly when PyTorch is missing. If PyTorch is installed but this table says CUDA is unavailable, the installed wheel is likely CPU-only or incompatible with the active Python/CUDA runtime.

In [ ]:
torch = None
try:
    import torch
except Exception as exc:
    display(Markdown(f"**PyTorch import failed or PyTorch is missing:** `{type(exc).__name__}: {exc}`"))

if torch is not None:
    torch_rows = [{
        "torch_version": torch.__version__,
        "torch_cuda_build": getattr(torch.version, "cuda", None),
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_count": torch.cuda.device_count(),
        "cudnn_version": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
    }]
    if torch.cuda.is_available():
        torch_rows[0]["device_0_name"] = torch.cuda.get_device_name(0)
        torch_rows[0]["device_0_capability"] = str(torch.cuda.get_device_capability(0))
    display(pd.DataFrame(torch_rows))

    if torch.cuda.is_available():
        x = torch.randn((2048, 2048), device="cuda")
        y = torch.randn((2048, 2048), device="cuda")
        torch.cuda.synchronize()
        started = time.perf_counter()
        z = x @ y
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        display(pd.DataFrame([{
            "tensor_device": str(z.device),
            "shape": str(tuple(z.shape)),
            "matmul_seconds": round(elapsed, 4),
            "allocated_gb": round(torch.cuda.memory_allocated() / 1024**3, 4),
            "reserved_gb": round(torch.cuda.memory_reserved() / 1024**3, 4),
        }]))
    else:
        display(Markdown("**PyTorch imported, but `torch.cuda.is_available()` is `False`. The PyTorch model cells will run on CPU."))

## 7. Tiny GPU Training Test

This is a minimal PyTorch training loop that confirms whether tensors, model parameters, and batches are actually placed on `cuda`. It runs only when CUDA is available.

In [ ]:
RUN_TINY_TRAINING_TEST = True

if torch is None:
    display(Markdown("Skipping tiny training test because PyTorch is not installed."))
elif not torch.cuda.is_available():
    display(Markdown("Skipping tiny training test because PyTorch CUDA is not available."))
elif not RUN_TINY_TRAINING_TEST:
    display(Markdown("Tiny training test is disabled."))
else:
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    device = torch.device("cuda")
    torch.manual_seed(42)

    X = torch.randn(12000, 64, dtype=torch.float32)
    true_w = torch.randn(64, 1, dtype=torch.float32)
    y = (X @ true_w + 0.1 * torch.randn(12000, 1)).squeeze()

    loader = DataLoader(TensorDataset(X, y), batch_size=1024, shuffle=True)
    model = nn.Sequential(
        nn.Linear(64, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 1),
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()

    before_memory = torch.cuda.memory_allocated()
    history = []
    for epoch in range(3):
        started = time.perf_counter()
        total_loss = 0.0
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(batch_X).squeeze()
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_X)
        torch.cuda.synchronize()
        history.append({
            "epoch": epoch + 1,
            "mean_loss": total_loss / len(X),
            "seconds": round(time.perf_counter() - started, 4),
            "model_device": str(next(model.parameters()).device),
        })

    after_memory = torch.cuda.memory_allocated()
    display(pd.DataFrame(history))
    display(pd.DataFrame([{
        "cuda_device": torch.cuda.get_device_name(0),
        "memory_before_mb": round(before_memory / 1024**2, 2),
        "memory_after_mb": round(after_memory / 1024**2, 2),
        "max_memory_allocated_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 2),
    }]))

## 8. Optional Installation Reference

This cell is disabled by default. Use it only after confirming that this notebook is running inside the workspace `.venv` kernel. For Windows, choose the current command from the official PyTorch selector if the command below becomes outdated: <https://pytorch.org/get-started/locally/>.

The existing final notebooks reference CUDA 12.8 wheels because NVIDIA drivers are backward-compatible with CUDA runtime versions bundled by PyTorch wheels. The CUDA version printed by `nvidia-smi` is the driver-supported maximum, not the exact CUDA runtime that PyTorch must use.

In [ ]:
INSTALL_CUDA_PYTORCH = False
PYTORCH_CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu128"

if INSTALL_CUDA_PYTORCH:
    install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        PYTORCH_CUDA_INDEX_URL,
    ]
    print("Running:", " ".join(install_command))
    subprocess.check_call(install_command)
else:
    print("Installation skipped. Set INSTALL_CUDA_PYTORCH = True only when you deliberately want this notebook to install PyTorch into the active kernel.")
    print("Manual command for this workspace, if appropriate:")
    print(r".venv\Scripts\python.exe -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128")

## 9. How to Interpret the Result

- If `nvidia-smi` works but `torch` is missing, the GPU is present but the Python training environment has no CUDA-enabled deep-learning backend.
- If `torch` imports but `torch.cuda.is_available()` is `False`, the installed PyTorch wheel is CPU-only, incompatible, or CUDA is hidden from the process.
- If the tiny training loop reports `model_device = cuda:0`, PyTorch GPU training works in this kernel.
- If assignment training still uses CPU after this test passes, the code path being run is probably a scikit-learn model or a backup/smoke setting is forcing CPU.
- `n_jobs=-1` in scikit-learn uses CPU parallelism, not the RTX 3080.